# Machine Unlearning Robustness Pipeline (Kaggle)

This unified notebook allows you to test the robustness of Machine Unlearning under Quantization right here in Kaggle, utilizing the provided GPU accelerators (T4 recommended).


## 1. Environment Initialization
We first need to install the project dependencies, including `bitsandbytes` for 4-bit/8-bit quantization and Hugging Face `accelerate`.

In [ ]:
!pip install -q -U torch transformers datasets bitsandbytes accelerate rouge-score

### Hugging Face Login
Gemma requires authentication. Ensure you have added your Hugging Face token as `HF_TOKEN` inside the **Kaggle Secrets** add-on menu.

In [ ]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("\n--- Authenticated with Hugging Face successfully! ---")
except Exception as e:
    print("\n--- Warning: Could not find HF_TOKEN in Kaggle Secrets ---")
    print("Please set it up via Add-ons -> Secrets to download Gemma.\n", e)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from torch.optim import AdamW
from rouge_score import rouge_scorer

MODEL_NAME = "google/gemma-3-1b"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Model and Dataset Operations

In [ ]:
def load_base_model():
    print(f"Loading Base Model {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return model, tokenizer

def load_muse_harry_potter():
    print("Loading MUSE Harry Potter subset...")
    dataset = load_dataset("muse-bench/MUSE", "harry_potter")
    forget_set = dataset['forget']
    retain_set = dataset['retain']
    print(f"Loaded {len(forget_set)} forget samples and {len(retain_set)} retain samples.")
    return forget_set, retain_set

## 3. Unlearning Implementations
These functions define Task Arithmetic and Gradient Ascent (Baseline NPO methods).

In [ ]:
def compute_task_vector(pretrained_model, finetuned_model):
    task_vector = {}
    with torch.no_grad():
        for name, base_param in pretrained_model.named_parameters():
            if name in finetuned_model.state_dict():
                task_vector[name] = finetuned_model.state_dict()[name] - base_param
    return task_vector

def apply_task_vector(pretrained_model, task_vector, scaling_factor=-1.0):
    print(f"Applying Task Vector with scaling={scaling_factor}")
    unlearned_model = AutoModelForCausalLM.from_pretrained(
        pretrained_model.config._name_or_path, 
        torch_dtype=torch.float16,
        device_map="auto"
    )
    with torch.no_grad():
        for name, param in unlearned_model.named_parameters():
            if name in task_vector:
                param.add_(task_vector[name] * scaling_factor)
    return unlearned_model

def gradient_ascent_unlearning(model, forget_dataloader, epochs=1, lr=1e-5):
    print("Executing standard Gradient Ascent...")
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr)
    for epoch in range(epochs):
        for batch in forget_dataloader:
            inputs, masks, labels = batch['input_ids'].to(model.device), batch['attention_mask'].to(model.device), batch['labels'].to(model.device)
            outputs = model(input_ids=inputs, attention_mask=masks, labels=labels)
            
            # Gradient ascent: maximize the standard loss -> minimize negative loss
            loss = -outputs.loss 
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
    return model

## 4. Post-Training Quantization (PTQ)
Loading the unlearned FP16 models in 4-bit to observe the Recovery Delta.

In [ ]:
def load_quantized_model(model_dir_path, bit_width=4):
    print(f"Quantizing model from {model_dir_path} to {bit_width}-bit precision...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=(bit_width == 4),
        load_in_8bit=(bit_width == 8),
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_dir_path,
        device_map="auto",
        quantization_config=bnb_config
    )
    return quantized_model

## 5. Evaluation Loop

In [ ]:
def evaluate_factual_recall(model, tokenizer, cloze_questions):
    correct = 0
    for question_dict in cloze_questions:
        prompt = question_dict['prompt']
        target_answer = question_dict['answer'].lower()
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=10)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        continuation = generated_text[len(prompt):].lower()
        if target_answer in continuation:
            correct += 1
            
    return correct / len(cloze_questions) if cloze_questions else 0

## Execution Sandbox
Run your components below to generate the metrics outlined in your proposal.

In [ ]:
# To begin the project:
# 1. Un-comment the loading code
# 2. Fine-tune your model on forget_set
# 3. Run Task Arithmetic and evaluate

# model, tokenizer = load_base_model()
# forget_set, retain_set = load_muse_harry_potter()

print("Ready to execute experiment pipeline!")